In [4]:
from unsloth import FastVisionModel
from datasets import load_dataset
from dotenv import load_dotenv
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from jiwer import wer, cer
from sklearn.model_selection import train_test_split
import json
import pandas as pd

True

In [ ]:
load_dotenv()

In [ ]:
# TODO: update with front/back details
field_structure = {
    "full_name": "string (arabic)",
    "national_id": "string, 14 digits",
    "address": "string (arabic)",
}

In [ ]:
SYSTEM_PROMPT = f'''
    You are a Vision Language Model tasked with extracted field values from an Egyptian national identity
    document. You must extract the fields without making any changes to the fields and return them
    as they are in Arabic script.
'''
USER_PROMPT = f'''
    Extract all the fields out of this Egyptian national ID following this format: {field_structure}.
    Return all fields as they appear and do not make any changes or updates to any of the fields. Return in
    in json format
'''

In [ ]:
def generate_conversation(data):
    image = data['image']
    full_name = data['full_name']
    national_id = data['national_id']
    address = data['address']
    message = f'full name: {full_name}, national_id: {national_id}, address: f{address}'
    conversation = [
        {
            'role': 'system',
            'content': [
                    {
                        'type': 'text',
                        'text': SYSTEM_PROMPT
                    }
            ]
        },
        {
            'role': 'user', 
            'content': [
                {
                    'type': 'text', 'text': USER_PROMPT 
                },
                {
                    'type': 'image', 
                    'image': image
                }
            ]
        },
        {
            'role': 'assistant', 
            'content': [
                {
                    'type': 'text', 
                    'text': message
                }
            ]
        }
    ]
    return conversation

##### Load and split data

In [ ]:
dataset = pd.read_csv("/path/to/dataset")

In [ ]:
# TODO: update to load images and fields separately
train, val = train_test_split(dataset, test_size=0.2)

#### Apply chat transformation

In [ ]:
training_data = []
for sample in train:
    training_data.append(generate_conversation(sample))

#### Define inference functions

In [ ]:
def infer(model, tokenizer, sample):
    FastVisionModel.for_inference(model)
    
    messages = [
        {
            'role': 'user',
            'content': [
                {
                    'type': 'image'
                }, 
                {
                    'type': 'text',
                    'text': USER_PROMPT
                }
            ]
        }
    ]

    input_text = tokenizer.apply_chat_templates(messages, add_generation_prompt=True)
    inputs = tokenizer(sample, input_text, add_special_tokens=False, return_tensors='pt').to(model.device)

    inference = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=1.5, min_p=0.1)

    return inference

In [ ]:
def batch_infer(model, tokenizer, samples):
    FastVisionModel.for_inference(model)
    predictions = []
    
    for sample in samples:
        prediction = infer(model, tokenizer, sample)
        predictions.append(prediction)
        
    return predictions


#### Define evaluation metrics

In [ ]:
def eval(model, tokenizer, samples):
    sample_images = [s['image'] for s in samples]
    predictions = batch_infer(model, tokenizer, sample_images)

    field_correct = {"full_name": 0, "national_id": 0, "address": 0}
    field_avg_wer = {"full_name": 0, "address": 0} # no words in national_id
    field_avg_cer = {"full_name": 0, "national_id": 0, "address": 0}

    total = len(samples)

    assert len(samples) == len(predictions)

    for sample, prediction in zip(samples, predictions):
        field_correct['full_name'] += 1 if prediction['full_name'] == sample['full_name'] else 0
        field_correct['national_id'] += 1 if prediction['national_id'] == sample['national_id'] else 0
        field_correct['address'] += 1 if prediction['address'] == sample['address'] else 0

        field_avg_wer['full_name'] += wer(prediction['full_name'], sample['full_name'])
        field_avg_wer['address'] += wer(prediction['address'], sample['address'])

        field_avg_cer['full_name'] += cer(prediction['full_name'], sample['full_name'])
        field_avg_cer['national_id'] += cer(prediction['national_id'], sample['national_id'])
        field_avg_cer['address'] += cer(prediction['address'], sample['address'])

    for key in field_correct:
        field_correct[key] /= total
        if key != 'national_id':
            field_avg_wer[key] /= total
        field_avg_cer[key] /= total

    return {"match": field_correct, "avg_cer": field_avg_cer, "avg_wer": field_avg_wer}
        

##### Load pretrained model

In [ ]:
# model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3-VL-8B-Instruct",
#                                                    load_in_4bit = False,
#                                                    use_gradient_checkpointing=True)


model, tokenizer = FastVisionModel.from_pretrained("unsloth/Qwen3.5-0.8B",
                                                   load_in_4bit = False,
                                                   use_gradient_checkpointing=True)

##### Pretrained model metrics (baseline)

In [ ]:
eval(model, tokenizer, val)

##### Set up finetuning model

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none"
)

In [ ]:
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=training_data,
    args=SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        learning_rate = 2e-4, #TODO tune these hyperparameters?
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        output_dir = "models",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048)
)


NameError: name 'model' is not defined

#### Train the model

In [ ]:
trainer_stats = trainer.train()

#### Save the model and tokenizer

In [ ]:
model.save_pretrained("qwen_vlm")
tokenizer.save_pretrained("qwen_vlm")